# Aula 21 — Planning and Goal Decomposition

**Quando escolher apenas a próxima ação deixa de ser suficiente?**

Na Aula 20, o agentic loop escolhia uma ação por vez a partir do estado observado.

Agora adicionamos uma nova capacidade:

> **construir, validar e revisar um plano explícito de múltiplas etapas.**

Regra da aula:

```text
planejar
≠
pensar mais

planejar
=
representar subobjetivos
+ dependências
+ pré-condições
+ ordem
+ revisão sob evidência
```

## 1. Objetivos

Ao final, você deverá conseguir:

- distinguir reactive next-action de planning explícito;
- decompor um objetivo em subtarefas;
- representar dependências;
- validar um plano antes da execução;
- separar plan generation de plan execution;
- observar replanning;
- medir overhead de planejamento;
- reconhecer quando o plano não traz utility adicional.

## 📘 Glossário da aula

- **[Planejamento](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#planejamento)**
- **[Decomposição de objetivo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#decomposição-de-objetivo)**
- **[Replanejamento](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#replanejamento)**
- **[Validação de plano](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#validação-de-plano)**

## 2. Reactive loop vs explicit planning

```text
Reactive
state → next action → execute → observe → repeat

Planning
goal → decompose → build plan → validate
     → execute → observe → continue / replan
```

Um plano não é “raciocínio escondido”.

Nesta aula ele será um **artefato estruturado e observável**.

In [ ]:
from dataclasses import dataclass, field
import pandas as pd

ACTIONS = {
    "inspect_request",
    "get_lesson_status",
    "build_release_note",
    "request_approval",
    "publish_simulated",
    "ask_human",
    "stop",
}

SCENARIOS = {
    "simple_ready": {
        "request_valid": True,
        "lesson_status": "ready",
        "approval_available": True,
        "status_changes_after_plan": False,
    },
    "not_ready": {
        "request_valid": True,
        "lesson_status": "review-candidate",
        "approval_available": True,
        "status_changes_after_plan": False,
    },
    "state_changes": {
        "request_valid": True,
        "lesson_status": "ready",
        "approval_available": True,
        "status_changes_after_plan": True,
    },
}

print("Cenários:", list(SCENARIOS))

## 3. Contrato do plano

Cada passo terá identidade, ação, dependências e motivo.

In [ ]:
@dataclass
class PlanStep:
    step_id: str
    action: str
    depends_on: list = field(default_factory=list)
    status: str = "pending"
    reason: str = ""

def build_plan():
    return [
        PlanStep("s1", "inspect_request", [], reason="validar pedido"),
        PlanStep("s2", "get_lesson_status", ["s1"], reason="observar prontidão"),
        PlanStep("s3", "build_release_note", ["s2"], reason="preparar release"),
        PlanStep("s4", "request_approval", ["s3"], reason="obter autorização"),
        PlanStep("s5", "publish_simulated", ["s4"], reason="publicar com aprovação"),
    ]

plan = build_plan()
display(pd.DataFrame([vars(x) for x in plan]))

## 4. Validação de plano

Antes da execução, validamos o artefato.

Critérios mínimos:

- action na allow-list;
- dependências existentes;
- nenhuma dependência aponta para o futuro;
- publicação depende de aprovação.

In [ ]:
def validate_plan(plan):
    ids = [s.step_id for s in plan]
    seen = set()
    errors = []

    for step in plan:
        if step.action not in ACTIONS:
            errors.append((step.step_id, "action_not_allowed"))

        for dep in step.depends_on:
            if dep not in ids:
                errors.append((step.step_id, f"missing_dependency:{dep}"))
            elif dep not in seen:
                errors.append((step.step_id, f"dependency_not_completed_before:{dep}"))

        if step.action == "publish_simulated":
            if "request_approval" not in [x.action for x in plan]:
                errors.append((step.step_id, "approval_step_missing"))

        seen.add(step.step_id)

    return errors

print("Plan errors:", validate_plan(plan))

### 4.1 Failure Lab — plano inválido

Vamos inverter duas etapas deliberadamente para observar uma falha de **plan quality** antes da execução.

In [ ]:
bad_plan = [
    PlanStep("s1", "inspect_request", []),
    PlanStep("s2", "get_lesson_status", ["s1"]),
    PlanStep("s3", "publish_simulated", ["s2"]),
    PlanStep("s4", "request_approval", ["s3"]),
]

display(pd.DataFrame([vars(x) for x in bad_plan]))
print("Plan errors:", validate_plan(bad_plan))

## 5. Baseline reativo

Reutilizamos o princípio da Aula 20: uma decisão por vez, sem plano persistente.

In [ ]:
def run_reactive(scenario):
    state = {
        "request_valid": scenario["request_valid"],
        "lesson_status": None,
        "approval": None,
        "published": False,
        "needs_human": False,
    }
    trace = []

    while True:
        if state["request_valid"] is not True:
            action = "ask_human"
        elif state["lesson_status"] is None:
            action = "get_lesson_status"
        elif state["lesson_status"] != "ready":
            action = "stop"
        elif state.get("release_note") is None:
            action = "build_release_note"
        elif state["approval"] is None:
            action = "request_approval"
        elif state["approval"] is not True:
            action = "ask_human"
        elif not state["published"]:
            action = "publish_simulated"
        else:
            action = "stop"

        if action == "get_lesson_status":
            state["lesson_status"] = scenario["lesson_status"]
        elif action == "build_release_note":
            state["release_note"] = "release simulada"
        elif action == "request_approval":
            state["approval"] = scenario["approval_available"]
        elif action == "publish_simulated":
            state["published"] = True
        elif action == "ask_human":
            state["needs_human"] = True
            trace.append(action)
            break
        elif action == "stop":
            trace.append(action)
            break

        trace.append(action)

    return state, trace

## 6. Execução orientada a plano

O executor não escolhe livremente a ordem. Ele percorre o plano validado e verifica se a evidência continua compatível.

In [ ]:
def run_plan_based(scenario):
    plan = build_plan()
    errors = validate_plan(plan)
    if errors:
        return {"published": False, "needs_human": False}, [], 0, errors

    state = {
        "request_valid": scenario["request_valid"],
        "lesson_status": None,
        "approval": None,
        "published": False,
        "needs_human": False,
    }

    trace = []
    replans = 0

    for i, step in enumerate(plan):
        action = step.action

        if action == "inspect_request" and state["request_valid"] is not True:
            state["needs_human"] = True
            trace.append("ask_human")
            break

        if action == "get_lesson_status":
            state["lesson_status"] = scenario["lesson_status"]

            if scenario.get("status_changes_after_plan"):
                state["lesson_status"] = "review-candidate"
                replans += 1
                trace.append("replan")
                trace.append("stop")
                break

        elif action == "build_release_note":
            if state["lesson_status"] != "ready":
                trace.append("stop")
                break
            state["release_note"] = "release simulada"

        elif action == "request_approval":
            state["approval"] = scenario["approval_available"]

        elif action == "publish_simulated":
            if state["approval"] is not True:
                state["needs_human"] = True
                trace.append("ask_human")
                break
            state["published"] = True

        trace.append(action)

    return state, trace, replans, errors

## 7. Comparative Evidence Lab

Agora executamos as duas arquiteturas sobre os mesmos cenários.

In [ ]:
rows = []

for name, scenario in SCENARIOS.items():
    r_state, r_trace = run_reactive(scenario)
    rows.append({
        "architecture": "reactive",
        "scenario": name,
        "success": r_state["published"],
        "actions": len(r_trace),
        "planning_steps": 0,
        "replans": 0,
        "human_intervention": r_state["needs_human"],
        "cost_proxy": len(r_trace),
    })

    p_state, p_trace, replans, errors = run_plan_based(scenario)
    rows.append({
        "architecture": "plan_based",
        "scenario": name,
        "success": p_state["published"],
        "actions": len(p_trace),
        "planning_steps": len(build_plan()),
        "replans": replans,
        "human_intervention": p_state["needs_human"],
        "cost_proxy": len(p_trace) + 0.5 * len(build_plan()) + replans,
    })

comparison = pd.DataFrame(rows)
display(comparison)

### Interpretação

Procure três situações:

1. **mesmo resultado + overhead de planejamento**  
   → planejar não conquistou sua complexidade;

2. **erro evitado antes da execução**  
   → plan validation trouxe valor;

3. **mudança de estado detectada**  
   → replanning ou safe stop podem evitar execução inválida.

> **Planning deve melhorar coordenação ou segurança de forma observável. Caso contrário, o loop reativo continua preferível.**

## 8. Architecture Decision Lab

Escolha entre:

```text
reactive next-action
explicit planning
deterministic workflow
```

Casos:

- A: sequência curta e estável;
- B: muitas dependências conhecidas;
- C: objetivo muda pouco, mas estado pode invalidar etapas futuras;
- D: tarefa de alto risco com ações irreversíveis.

Justifique por:

- variabilidade;
- dependências;
- reversibilidade;
- necessidade de inspeção prévia;
- custo de planning;
- risco de stale plan.

In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução de referência\n\n{self._solution_text}"))

q21_arch = TILExercise(
    "Pergunte se há dependências suficientes para justificar um plano explícito e se o estado pode invalidar etapas futuras.",
    "A tende a workflow/reactive simples; B favorece planning; C favorece planning com validação/replanning; D requer planning governado e forte supervisão humana.",
)

print("Use q21_arch.hint() ou q21_arch.solution() somente quando desejar.")

## 9. Checagem de compreensão

1. Qual a diferença entre plan generation e plan execution?
2. Por que um plano deve ser validado antes de executar?
3. Quando replanning é justificável?
4. Por que um plano explícito não é chain-of-thought?
5. Em qual cenário o planejamento adicionou apenas overhead?
6. Que falha pode ser evitada antes de qualquer tool call?
7. Quando um workflow determinístico ainda é melhor?

## 10. Reprodutibilidade

```text
Internet OFF
GPU OFF
sem API externa
sem side effects reais
planner determinístico
plan artifact observável
```

## 11. Síntese

```text
goal
→ decomposition
→ explicit plan
→ validation
→ execution
→ observation
→ continue / replan / stop
```

> **Planejamento é útil quando a estrutura explícita de dependências e pré-condições melhora coordenação ou segurança o suficiente para justificar seu overhead.**

---

## Continue no TIL

← **[Anterior: Aula 20 — Agentic Systems Foundations](https://www.kaggle.com/code/pedrogentil/til-20-agentic-systems-foundations)** &nbsp;&nbsp;|&nbsp;&nbsp; 🏠 **[Apresentação do curso](https://www.kaggle.com/code/pedrogentil/text-intelligence-lab-course)** &nbsp;&nbsp;|&nbsp;&nbsp; **Próxima unidade: Agentic Systems — em preparação** →